# Feature combination benchmark

New feature engineering is paused. This notebook reads saved evidence only; it never starts model fitting. Scores below are development diagnostics, not Kaggle scores. No candidate is promoted automatically.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from scripts.feature_combination_evaluation import figures, review_figures
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/feature_combination_evaluation.json").is_file())
PUBLIC = ROOT / "reports/feature_combinations"
review = json.loads((PUBLIC / "review.json").read_text())
print("Historical first-submission count:", review["first_submission"])
print("Planned candidate configurations:", len(review["plan"]))

Historical first-submission count: {'word_unigram_bigram_columns': 40000, 'explicit_similarity_columns': 8, 'total_columns': 40008, 'classifier_fits': 0, 'purpose': 'historical feature count only, not validation'}
Planned candidate configurations: 49


## Evaluation boundary

Training and query text groups are disjoint. Existing label-dependent reference banks were cross-fitted. They are **not** safe as precomputed inputs for a new CV split: inner training rows could contain statistics derived from the held-out fold. Fresh nested validation must rebuild reference pools, vocabularies, supervised weighting, screening and scaling inside each split. The current adapted training margin is in-sample; two explicit frozen-margin sensitivity models examine that dependency without claiming fresh confirmation.

In [2]:
result = json.loads((PUBLIC / "results.json").read_text())
print(result["status"], "| Completed configurations:", len(result["completed_variants"]))
display(pd.DataFrame(result["models"]).sort_values("macro_auc", ascending=False))
CHARTS = figures(result)

COMBINATION_SCREEN_COMPLETE | Completed configurations: 49


,variant,advertising_auc,legal_advice_auc,macro_auc,pooled_auc,ranked_pooled_auc,mean_brier,mean_log_loss
3,context_evidence,0.703470,0.755044,0.729257,0.736183,0.741330,0.241230,0.831778
7,anchor_refit,0.703470,0.755044,0.729257,0.736183,0.741330,0.241230,0.831778
4,uniform_all,0.703246,0.754574,0.728910,0.735914,0.740913,0.240838,0.819310
16,add_actor_support,0.703470,0.754134,0.728802,0.735392,0.740687,0.241282,0.828059
11,add_relations,0.688246,0.768106,0.728176,0.745248,0.746836,0.245913,0.884918
22,add_windows,0.702500,0.753439,0.727970,0.735951,0.739906,0.244988,0.857383
45,pair_consistency_actor_support,0.704590,0.750347,0.727468,0.732765,0.738160,0.243549,0.829029
14,add_matched_pairs,0.695933,0.757979,0.726956,0.739120,0.741499,0.241776,0.831899
23,add_lexical,0.697276,0.755934,0.726605,0.736098,0.740312,0.244040,0.848811
19,add_consistency,0.704590,0.748606,0.726598,0.730519,0.736879,0.244808,0.838613


## Per-policy AUC and conditional uncertainty

A mean gain cannot conceal a policy regression. Partial-run intervals are not final, and no interval removes the earlier adaptive search.

In [3]:
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")

## Marginal and conditional family contributions

Add-one tests start at the common context reference; removal tests start from the full union. Coefficients alone are not treated as feature value.

In [4]:
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

## Pairs and distinct metric definitions

Pairs are fixed before this run. Different encodings and hyperparameters are separate model configurations, not silently selected winners.

In [5]:
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

## Probability quality

AUC, Brier and log loss measure different properties. No calibration is fitted on evaluation targets.

In [6]:
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")

## Model size and sensitivity checks

All-feature matrices retain eligible nonconstant, nonduplicate dense columns and training-observed sparse columns. Raw column counts are not evidence of quality.

In [7]:
CHARTS[8].show(renderer="plotly_mimetype")
CHARTS[9].show(renderer="plotly_mimetype")

## Decision and required confirmation

The all-feature C=1 model is the prespecified primary diagnostic, not an expected winner. Report every candidate and regression. Any later shortlist or hyperparameter choice must be selected inside properly nested, group-safe validation. Do not call a re-split of previously inspected comments an untouched holdout. For Kaggle comparison, freeze one inference-ready specification and distinguish its true submission score from this table.

In [8]:
print("Promotion:", result["selection"])
for limitation in result["limitations"]:
    print(limitation)
display(pd.DataFrame(result["design_sizes"]))

Promotion: NONE_AUTOMATIC_DEVELOPMENT_ONLY
Development screen only: the same 881 comments informed previous research decisions.
Conditional bootstrap intervals do not undo historical selection bias or refit encoders.
Adapted training answer margins are in-sample; frozen-margin sensitivity is separate.
Do not use cached supervised banks as precomputed inputs to a new cross-validation split.
Before promotion, rebuild every supervised transform inside nested group-safe folds.
Only two policies are observed; these results do not estimate unseen-policy performance.
Early sparse variants use new same-rule training fits here, not their old pooled fits.
No new Kaggle score, independent confirmation, or automatic model promotion is produced.


,fold,variant,dense_input_columns,dense_retained_columns,sparse_columns
0,0,all_features,728,447,20356
1,1,all_features,728,527,14006
2,0,all_features_stronger_penalty,728,447,20356
3,1,all_features_stronger_penalty,728,527,14006
4,0,anchor_refit,34,22,0
...,...,...,...,...,...
93,1,compact_lexical_evidence,260,204,14006
94,0,all_lexical_scope,728,447,25347
95,1,all_lexical_scope,728,527,19503
96,0,all_lexical_scope_copy,728,447,25000
